# Task 3 — Local / Upstream Divergence Distribution

For each variable with both a local (`_s`) and upstream (`_u`) value, compute the global distribution of divergence across all 190k L8 basins. Key questions:
- Which pairs show the strongest divergence globally? The least?
- Is strong divergence rare (a specific environmental signal) or common?
- Where do reference sites (Timbuktu, Ur, Kaifeng) fall in each divergence distribution?

**Divergence metric**: `log2(u/s)` for ratio pairs (symmetric: 0 = identical, +1 = upstream 2× local, −1 = local 2× upstream); `u − s` for difference pairs (variables near zero or with negative values where ratio is unstable).

See `docs/edop/data_exploration.md` Task 3 for full specification.

In [1]:
# Cell 1 — Imports and connection
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys

sys.path.insert(0, '/Users/karlg/Documents/Repos/_cedop')
from scripts.shared.db_utils import db_connect

conn = db_connect()
print("connected")

connected


In [2]:
# Cell 2 — S/U pair definitions
#
# Columns: (api_key, s_col, u_col, units, method, scale_factor)
#   method='ratio'  → divergence = log2(u/s), computed where s>0 and u>0
#   method='diff'   → divergence = u - s  (used where near-zero or negative values
#                      make ratios unstable: temp, human footprint, sparse land-cover)
#   scale_factor    → multiply raw DB value before analysis (temp stored ×10 → 0.1)

SU_PAIRS = [
    # Climate
    ('aridity',      'ari_ix_sav', 'ari_ix_uav', 'P/PET×100', 'ratio', 1.0),
    ('precip_yr',    'pre_mm_syr', 'pre_mm_uyr', 'mm/yr',     'ratio', 1.0),
    ('temp_yr',      'tmp_dc_syr', 'tmp_dc_uyr', '°C',        'diff',  0.1),
    # Terrain
    ('slope',        'slp_dg_sav', 'slp_dg_uav', 'degrees',   'ratio', 1.0),
    # Hydrology
    ('river_area',   'ria_ha_ssu', 'ria_ha_usu', 'ha',        'ratio', 1.0),
    ('wet_pct_g1',   'wet_pc_sg1', 'wet_pc_ug1', '%',         'diff',  1.0),
    ('karst',        'kar_pc_sse', 'kar_pc_use', '%',         'diff',  1.0),
    # Land use / Human
    ('cropland',     'crp_pc_sse', 'crp_pc_use', '%',         'diff',  1.0),
    ('human_fp_09',  'hft_ix_s09', 'hft_ix_u09', 'index',     'diff',  1.0),
]

print(f"{len(SU_PAIRS)} s/u pairs defined")
for row in SU_PAIRS:
    print(f"  {row[0]:15s}  {row[1]} / {row[2]}  [{row[4]}]")

9 s/u pairs defined
  aridity          ari_ix_sav / ari_ix_uav  [ratio]
  precip_yr        pre_mm_syr / pre_mm_uyr  [ratio]
  temp_yr          tmp_dc_syr / tmp_dc_uyr  [diff]
  slope            slp_dg_sav / slp_dg_uav  [ratio]
  river_area       ria_ha_ssu / ria_ha_usu  [ratio]
  wet_pct_g1       wet_pc_sg1 / wet_pc_ug1  [diff]
  karst            kar_pc_sse / kar_pc_use  [diff]
  cropland         crp_pc_sse / crp_pc_use  [diff]
  human_fp_09      hft_ix_s09 / hft_ix_u09  [diff]


In [3]:
# Cell 3 — Check column availability and fetch all s/u columns from basin08

def get_existing_cols(conn, table):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT column_name FROM information_schema.columns
            WHERE table_schema='public' AND table_name=%s
        """, (table,))
        return {row[0] for row in cur.fetchall()}

existing = get_existing_cols(conn, 'basin08')

all_cols = []
for api_key, s_col, u_col, *_ in SU_PAIRS:
    for col in [s_col, u_col]:
        if col in existing:
            all_cols.append(col)
        else:
            print(f"  MISSING in basin08: {col}")

col_sql = ', '.join(f'"{c}"' for c in all_cols)
with conn.cursor() as cur:
    cur.execute(f'SELECT {col_sql} FROM public.basin08')
    rows = cur.fetchall()
    df_raw = pd.DataFrame(rows, columns=all_cols)

df_raw = df_raw.replace(-9999, np.nan)
print(f"Loaded {len(df_raw):,} rows × {len(df_raw.columns)} columns")
print(df_raw.describe().round(2))

Loaded 190,675 rows × 18 columns
       ari_ix_sav  ari_ix_uav  pre_mm_syr  pre_mm_uyr  tmp_dc_syr  tmp_dc_uyr  \
count   190675.00   190675.00   190675.00   190675.00   190675.00   190675.00   
mean        95.34      100.59      790.90      797.67      124.56      119.04   
std        169.61      186.82      735.72      724.19      135.90      135.66   
min          0.00        0.00        0.00        0.00     -243.00     -243.00   
25%         27.00       30.00      277.00      293.00       18.00       14.00   
50%         68.00       70.00      555.00      570.00      167.00      159.00   
75%        106.00      108.00     1108.00     1119.00      246.00      241.00   
max       2167.00     2167.00     8446.00     7592.00      314.00      314.00   

       slp_dg_sav  slp_dg_uav  ria_ha_ssu  ria_ha_usu  wet_pc_sg1  wet_pc_ug1  \
count   184285.00   184285.00   190675.00   190675.00   190675.00   190675.00   
mean        41.73       46.79      192.74     8171.95       10.19        7.

In [4]:
# Cell 4 — Apply scale factors; compute divergence series for each pair
#
# divergence[api_key] = pd.Series of per-basin divergence values (non-null only)
# raw_pairs[api_key]  = (s_series, u_series) after scaling, full length

divergence = {}
raw_pairs  = {}

for api_key, s_col, u_col, units, method, scale in SU_PAIRS:
    if s_col not in df_raw.columns or u_col not in df_raw.columns:
        print(f"  skipping {api_key} (column missing)")
        continue

    s = df_raw[s_col] * scale
    u = df_raw[u_col] * scale

    if method == 'ratio':
        mask = (s > 0) & (u > 0) & s.notna() & u.notna()
        div  = np.log2(u[mask] / s[mask])
    else:
        mask = s.notna() & u.notna()
        div  = u[mask] - s[mask]

    divergence[api_key] = div
    raw_pairs[api_key]  = (s, u)

    print(f"{api_key:16s}  n={len(div):,}  "
          f"median={div.median():+.3f}  "
          f"p05={div.quantile(0.05):+.3f}  "
          f"p95={div.quantile(0.95):+.3f}")

aridity           n=187,368  median=+0.000  p05=-0.150  p95=+0.555
precip_yr         n=189,948  median=+0.000  p05=-0.159  p95=+0.393
temp_yr           n=190,675  median=+0.000  p05=-3.130  p95=+0.200
slope             n=174,761  median=+0.000  p05=-0.663  p95=+2.322
river_area        n=186,333  median=+0.000  p05=-0.000  p95=+6.679
wet_pct_g1        n=190,675  median=+0.000  p05=-28.000  p95=+3.000
karst             n=190,675  median=+0.000  p05=-1.000  p95=+13.000
cropland          n=190,675  median=+0.000  p05=-9.000  p95=+6.000
human_fp_09       n=190,675  median=+0.000  p05=-38.000  p95=+23.000


In [5]:
# Cell 5 — Summary statistics table, ranked by absolute median divergence

rows = []
for api_key, s_col, u_col, units, method, scale in SU_PAIRS:
    if api_key not in divergence:
        continue
    div = divergence[api_key]
    rows.append({
        'api_key':    api_key,
        'units':      units,
        'method':     method,
        'n_valid':    len(div),
        'median_div': round(div.median(), 3),
        'p25':        round(div.quantile(0.25), 3),
        'p75':        round(div.quantile(0.75), 3),
        'p95':        round(div.quantile(0.95), 3),
        'p99':        round(div.quantile(0.99), 3),
        'p01':        round(div.quantile(0.01), 3),
        'p05':        round(div.quantile(0.05), 3),
        'pct_upstream_greater': round((div > 0).mean() * 100, 1),
        'abs_med':    round(div.abs().median(), 3),
    })

summary = pd.DataFrame(rows).sort_values('abs_med', ascending=False)
print(summary[['api_key','method','median_div','p05','p95','pct_upstream_greater','abs_med']].to_string(index=False))

    api_key method  median_div     p05    p95  pct_upstream_greater  abs_med
    aridity  ratio         0.0  -0.150  0.555                  30.6      0.0
  precip_yr  ratio         0.0  -0.159  0.393                  31.4      0.0
    temp_yr   diff         0.0  -3.130  0.200                   8.2      0.0
      slope  ratio         0.0  -0.663  2.322                  31.0      0.0
 river_area  ratio         0.0  -0.000  6.679                  44.5      0.0
 wet_pct_g1   diff         0.0 -28.000  3.000                   9.9      0.0
      karst   diff         0.0  -1.000 13.000                  11.1      0.0
   cropland   diff         0.0  -9.000  6.000                  13.4      0.0
human_fp_09   diff         0.0 -38.000 23.000                  19.8      0.0


In [6]:
# Cell 6 — Save summary CSV

out_dir = '/Users/karlg/Documents/Repos/_cedop/output/edop/explore'
os.makedirs(out_dir, exist_ok=True)

summary.to_csv(f'{out_dir}/03_su_divergence_summary.csv', index=False)
print(f"Saved 03_su_divergence_summary.csv")

Saved 03_su_divergence_summary.csv


In [7]:
# Cell 7 — ECDF plots for all pairs
#
# ECDF (Empirical Cumulative Distribution Function): x-axis = divergence value,
# y-axis = fraction of basins with divergence ≤ x.
# Vertical dashed line at 0 = no divergence.
# Red line = median. The curve's shape shows how common large divergences are.

n_pairs = len(divergence)
ncols   = 3
nrows   = (n_pairs + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.5))
axes = axes.flatten()

for i, (api_key, s_col, u_col, units, method, scale) in enumerate(SU_PAIRS):
    if api_key not in divergence:
        continue
    ax  = axes[i]
    div = divergence[api_key]

    sorted_div = np.sort(div.values)
    ecdf       = np.arange(1, len(sorted_div) + 1) / len(sorted_div)

    ax.plot(sorted_div, ecdf, lw=1.2, color='steelblue')
    ax.axvline(0,            color='black', lw=0.8, ls='--', alpha=0.6, label='zero')
    ax.axhline(0.5,          color='gray',  lw=0.5, ls=':')
    ax.axvline(div.median(), color='tomato', lw=1.0, ls='-', alpha=0.9, label='median')

    ax.set_title(api_key, fontsize=9)
    xlabel = 'log₂(u/s)' if method == 'ratio' else f'u − s  ({units})'
    ax.set_xlabel(xlabel, fontsize=7)
    ax.set_ylabel('ECDF', fontsize=7)
    ax.tick_params(labelsize=7)

    med = div.median()
    ax.text(0.97, 0.05, f'n={len(div):,}\nmed={med:+.2f}',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=6.5, color='tomato')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('S/U Divergence — ECDF (L8, ~190k basins)', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(f'{out_dir}/03_su_divergence_ecdf.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved 03_su_divergence_ecdf.png")

Saved 03_su_divergence_ecdf.png


In [8]:
# Cell 8 — Reference place basin lookup
#
# Three sites with known environmental character:
#   Timbuktu: hyper-arid local, Niger River upstream moisture
#   Ur:       hyper-arid local, Tigris/Euphrates highland source
#   Kaifeng:  semi-arid Yellow River basin, moderate divergence expected
#
# find_main_basin: within 25km, pick basin with largest upstream area
# (same logic as the sandbox basin assignment)

PLACES = {
    'Timbuktu': (16.766,  -3.008),
    'Ur':       (30.962,  46.103),
    'Kaifeng':  (34.797, 114.341),
}

def find_main_basin(conn, lat, lon, radius_m=25000):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT hybas_id, up_area
            FROM public.basin08
            WHERE ST_DWithin(
                geom::geography,
                ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography,
                %s
            )
            ORDER BY up_area DESC
            LIMIT 1
        """, (lon, lat, radius_m))
        return cur.fetchone()

def get_basin_vals(conn, hybas_id, cols):
    col_sql = ', '.join(f'"{c}"' for c in cols)
    with conn.cursor() as cur:
        cur.execute(f'SELECT {col_sql} FROM public.basin08 WHERE hybas_id = %s', (hybas_id,))
        row = cur.fetchone()
    return dict(zip(cols, row)) if row else {}

fetch_cols = [c for _, s_col, u_col, *_ in SU_PAIRS for c in [s_col, u_col]]
place_data = {}

for place, (lat, lon) in PLACES.items():
    result = find_main_basin(conn, lat, lon)
    if result is None:
        print(f"{place}: no basin found")
        continue
    hybas_id, up_area = result
    vals = get_basin_vals(conn, hybas_id, fetch_cols)
    place_data[place] = {'hybas_id': hybas_id, 'up_area_km2': up_area, 'vals': vals}
    print(f"{place:12s}  hybas_id={hybas_id}  up_area={up_area:,.0f} km²")

Kaifeng       hybas_id=4080602410.0  up_area=734,701 km²


In [9]:
# Cell 9 — Compute divergence values and global percentile rank for each place
#
# percentile = fraction of all basins with lower divergence than this place.
# p99 means only 1% of basins show more divergence — a genuinely extreme signal.

pct_rows = []

for place, info in place_data.items():
    brow = info['vals']
    for api_key, s_col, u_col, units, method, scale in SU_PAIRS:
        if api_key not in divergence:
            continue
        s_raw = brow.get(s_col)
        u_raw = brow.get(u_col)

        if s_raw is None or u_raw is None:
            div_val = pct = None
        else:
            s_val = s_raw * scale
            u_val = u_raw * scale
            if method == 'ratio':
                div_val = np.log2(u_val / s_val) if (s_val > 0 and u_val > 0) else None
            else:
                div_val = u_val - s_val
            if div_val is not None:
                arr = divergence[api_key].values
                pct = float((arr < div_val).mean() * 100)
            else:
                pct = None

        pct_rows.append({
            'place':    place,
            'api_key':  api_key,
            'method':   method,
            'units':    units,
            'div_value': round(div_val, 3) if div_val is not None else None,
            'percentile': round(pct, 1)    if pct  is not None else None,
        })

pct_df = pd.DataFrame(pct_rows)

# Print by place, sorted by absolute divergence
for place in PLACES:
    sub = pct_df[(pct_df.place == place) & pct_df.percentile.notna()].copy()
    sub['abs_pct_from_median'] = (sub.percentile - 50).abs()
    sub = sub.sort_values('abs_pct_from_median', ascending=False)
    print(f"\n{'='*55}")
    print(f"{place}  (hybas_id={place_data[place]['hybas_id']}, "
          f"up_area={place_data[place]['up_area_km2']:,.0f} km²)")
    print(f"{'='*55}")
    print(sub[['api_key','method','div_value','percentile']].to_string(index=False))


Timbuktu  (hybas_id=1080561810.0, up_area=379,818 km²)
    api_key method  div_value  percentile
  precip_yr  ratio      2.369        99.9
    aridity  ratio      2.385        99.8
 river_area  ratio      8.764        98.7
      slope  ratio      3.585        97.9
 wet_pct_g1   diff    -56.000         2.2
human_fp_09   diff     31.000        96.7
   cropland   diff      9.000        96.4
      karst   diff      1.000        88.9
    temp_yr   diff     -1.600        11.4

Ur  (hybas_id=2080818060.0, up_area=456,772 km²)
    api_key method  div_value  percentile
   cropland   diff    -45.000         0.4
    aridity  ratio      2.070        99.6
  precip_yr  ratio      1.457        99.5
    temp_yr   diff     -6.000         1.6
human_fp_09   diff    -64.000         2.5
 wet_pct_g1   diff    -46.000         3.0
      karst   diff     16.000        95.6
 river_area  ratio      6.437        94.3

Kaifeng  (hybas_id=4080602410.0, up_area=734,701 km²)
    api_key method  div_value  percentile

In [10]:
# Cell 10 — Save place percentile table and annotated ECDF with place markers

pct_df.to_csv(f'{out_dir}/03_su_place_percentiles.csv', index=False)
print("Saved 03_su_place_percentiles.csv")

# Annotated ECDF: overlay place markers on each pair's curve
PLACE_STYLES = {
    'Timbuktu': ('tomato',  'o'),
    'Ur':       ('goldenrod', 's'),
    'Kaifeng':  ('mediumseagreen', '^'),
}

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.5))
axes = axes.flatten()

for i, (api_key, s_col, u_col, units, method, scale) in enumerate(SU_PAIRS):
    if api_key not in divergence:
        continue
    ax  = axes[i]
    div = divergence[api_key]

    sorted_div = np.sort(div.values)
    ecdf       = np.arange(1, len(sorted_div) + 1) / len(sorted_div)

    ax.plot(sorted_div, ecdf, lw=1.2, color='steelblue', alpha=0.8)
    ax.axvline(0,            color='black', lw=0.7, ls='--', alpha=0.5)
    ax.axhline(0.5,          color='gray',  lw=0.5, ls=':')

    for place, (color, marker) in PLACE_STYLES.items():
        sub = pct_df[(pct_df.place == place) & (pct_df.api_key == api_key)]
        if sub.empty or sub.iloc[0]['div_value'] is None:
            continue
        dv  = sub.iloc[0]['div_value']
        pct = sub.iloc[0]['percentile'] / 100
        ax.plot(dv, pct, marker=marker, color=color, ms=7, zorder=5, label=place)

    ax.set_title(api_key, fontsize=9)
    xlabel = 'log₂(u/s)' if method == 'ratio' else f'u − s  ({units})'
    ax.set_xlabel(xlabel, fontsize=7)
    ax.set_ylabel('ECDF', fontsize=7)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=6.5, loc='upper left')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('S/U Divergence ECDF with reference sites (L8)', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(f'{out_dir}/03_su_divergence_ecdf_annotated.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved 03_su_divergence_ecdf_annotated.png")

Saved 03_su_divergence_ecdf_annotated.png
